## Filesystem Hierarchy Standard

1. `/`: **root directory**, the root of the **entire system**.
2. `/home`: where user files are stored, e.g. `/home/dennie`. This is where are stored documents, downloads, pictures, videos, projects and everythin related to **your account** - personal workspace. Every user gets their **own `/home`** directory in order to keep files separate from other users on the system.
3. `/root`: is **NOT** the root of the file system, it's the **home directory of the administrator** account, know as the **root user**. It's just one folder inside `/` - root directory of the filesystem.
4. `/bin` (binaries): This folder contains essential **command line programs** that Linux **needs to function**. It contains commands like `ls`, `cp`, `mv`, `cat`, `mkdir`, etc. Think of `/bin` as the toolbox that allows Linux to operate.
5. `/sbin`(system binaries): contains administrative tools used for managing the operating system. Networking (`ip`, `ifconfig`, `route`), Disk management (`fdisk`, `mkfs`, `parted`), Booting the system (`grub-install`, `init`), System Recovery (`fsck`, `e2fsck`).

Most regular users don't use these commands every day, but **administrators** do.

6. `/etc`: where Linux stores **configuration files**. **The Settings folder for the entire operating system**. Almost every application has a configuration file, somewhere inside `/etc`. Network settings, SSH and DNS settings all live in `/etc`. Edit a configuration. Instead of storing a configuration file insdie the application itself, Linux keeps most settings here.
7. `/usr` (Unix System Resources): This folder contains **most of the software installed** on your system. The **main software library** of your machine. Applications, Documentations, Libraries, Documentation, Development tools, Shared Resources.
```bash
ls /usr/bin | wc -l
809
```
Displays a number of executable programs.
8. `/lib`: Applications also need **shared code**. Libraries are **reusable pieces of code** that many applications share. Instead of every program including the same code repeatedly, Linux **stores common functionality in shared libraries**. This saves **disk space** and makes **updates much easier**. Think of libraries as **shared building blocks** that applications borrow whenever they need them.

9. `/tmp`: is used for **temporary data**. Applications create files here while they are running. Many Linux systems automatically clean this directory diring reboot. **Don't store ANY thing IMPORTANT** inside `/tmp` unless you are ok with losing it.

10. `/var` (variable): contains **files that constantly grow and change**. `/var/log` system logs, `/var/mail` mail queues, `/var/lib` libraries, databases, cache files, printer queues. Whenever you are troubleshooting Linux, `/var/log` is one of the first places you check, because it contains log files that explain what the system has been doing. Suppose you plug in a USB drive. Where does it appear? Usually in directories like `/media` or `/mnt`

In [ ]:
tail /var/log/syslog
2026-08-22T11:02:00.583043+00:00 HP systemd[1]: wsl-pro.service: Deactivated successfully.
2026-08-22T11:03:26.862531+00:00 HP systemd[1]: apt-daily.service: Deactivated successfully.
2026-08-22T11:03:26.863021+00:00 HP systemd[1]: Finished apt-daily.service - Daily apt download activities.
2026-08-22T11:03:26.863292+00:00 HP systemd[1]: apt-daily.service: Consumed 2min 38.864s CPU time.
2026-08-22T11:04:21.697366+00:00 HP PackageKit: daemon quit
2026-08-22T11:04:21.709018+00:00 HP systemd[1]: packagekit.service: Deactivated successfully.
2026-08-22T11:17:01.883694+00:00 HP CRON[29708]: (root) CMD (cd / && run-parts --report /etc/cron.hourly)

11. `/media`: is commonly used for **automatically mounted devices**, such as USB drive and DVDs.
12. `/mnt`: is used by administrators when **manually mounting disks**, **network shares** or other **storage devices**. This is another major difference from Windows. Instead of assigning a drive letter (e.g. `D:\`), Linux attaches **the new storage** somewhere within the existing directory tree.
13. `/dev`: in Linux almost everyting is treated as a file, hard drives, USB devices, keyboards, mice, printers, terminals, they **all appear as files** inside `/dev`. For example, you hard drive might appear as `/dev/sda`:

In [ ]:
/dev/sda
brw-rw---- 1 root disk 8, 0 Aug 21 21:48 /dev/sda

14. **`/srv`** ("service") holds data this host **serves out** to others, as opposed to `/var`, which holds data the system generates for its own internal use (logs, caches, spool). Historically it's where you'd find FTP/HTTP content roots (`/srv/www`, `/srv/ftp`), git repos being hosted (`/srv/git`), or — like here — data whose whole purpose is to be pulled off this box by something else (a backup retrieval process, another server, an offsite sync job). It's the least strictly standardized of the FHS directories in practice — many admins just use it loosely as "host-specific data trees, not internal system state" — but the `/var` vs `/srv` distinction (internal-use vs. served-to-others) is the intent behind the split.

One **file-based interface** for hardware and files alike. It allows programs to interact with hardware using **the same file-base interface** they use for regular files. It's one of the reasons Linux is so flexible.
14. `/proc`: it doesn't even exist on your hard disk. It's a **virtual filesystem** created by the Linux kernel. It contains **live information about running processes**, CPU usage, memory, system configuration and much more. Monitoring tools need info about your system, they often read it from `/proc`:

In [ ]:
cat /proc/meminfo
MemTotal:        8055396 kB
MemFree:         5030064 kB
MemAvailable:    6409356 kB
Buffers:           29944 kB
Cached:          1466988 kB
SwapCached:            0 kB
Active:           468564 kB
Inactive:        2210212 kB

Think of it as a live dashboard of what's happening inside Linux right now.
15. `/boot`: contains everything needed to start Linux. The Linux kernel, boo loader files and startup configuration all live here. Without these files, your operating system wouldn't be able to boot.

---

## PART 1: The Linux Filesystem Is a Single Tree — Not Drive Letters

**Support context:** In Windows you think in drives — `C:\`, `D:\`. In Linux, there is **one tree**, starting at `/` (root). Everything — disks, USB drives, network shares, even your Windows C: drive inside WSL — gets _mounted_ somewhere into that one tree.
> When a user says "the app can't find its config" or "disk is full," your first move is always: **where in the tree am I**, and **what's mounted where**. There's no "which drive" question in Linux — there's only "which path."

**Developer bridge:** Think of it like a single Django project's `settings.py` — one `BASE_DIR`, **everything else is a path relative to it**. Except here, `BASE_DIR` is `/`, and literally everything on the machine — including other physical disks — hangs off that one root. **Your `/mnt/c` in WSL is Windows' entire `C:\` drive**, _mounted into_ the Linux tree at that path. It's not a separate namespace — it's just a folder like any other, as far as commands are concerned.

**Why this matters for support:** "Everything is a file, everything is reachable from `/`" is the single biggest mental shift from Windows. Devices (`/dev`), running processes (`/proc`), kernel settings (`/sys`) — all appear as paths in the same tree you `cd` into. No separate "Device Manager" GUI required — you `cat` and `ls` your way through actual hardware/kernel state.

---

In [ ]:
dennie@HP:~/linux-tut$ pwd
/home/dennie/linux-tut

In [ ]:
dennie@HP:~/linux-tut$ ls -la /
total 2856
drwxr-xr-x  22 root root    4096 Aug 21 21:48 .
drwxr-xr-x  22 root root    4096 Aug 21 21:48 ..
lrwxrwxrwx   1 root root       7 Apr 22  2024 bin -> usr/bin
drwxr-xr-x   2 root root    4096 Feb 26  2024 bin.usr-is-merged
drwxr-xr-x   2 root root    4096 Apr 22  2024 boot
drwxr-xr-x  15 root root    3840 Aug 21 21:48 dev
drwxr-xr-x  88 root root    4096 Aug 22 14:07 etc
drwxr-xr-x   3 root root    4096 Aug 21 21:33 home
-rwxr-xr-x   1 root root 2836528 Aug 17 18:44 init
lrwxrwxrwx   1 root root       7 Apr 22  2024 lib -> usr/lib
drwxr-xr-x   2 root root    4096 Apr  8  2024 lib.usr-is-merged
lrwxrwxrwx   1 root root       9 Apr 22  2024 lib64 -> usr/lib64
drwx------   2 root root   16384 Aug 21 21:31 lost+found
drwxr-xr-x   2 root root    4096 Feb 10  2026 media
drwxr-xr-x   6 root root    4096 Aug 21 21:32 mnt
drwxr-xr-x   2 root root    4096 Feb 10  2026 opt
dr-xr-xr-x 212 root root       0 Aug 21 21:48 proc
drwx------   3 root root    4096 Feb 10  2026 root
drwxr-xr-x  20 root root     600 Aug 22 11:03 run
lrwxrwxrwx   1 root root       8 Apr 22  2024 sbin -> usr/sbin
drwxr-xr-x   2 root root    4096 Mar 31  2024 sbin.usr-is-merged
drwxr-xr-x   2 root root    4096 Aug 21 21:32 snap
drwxr-xr-x   2 root root    4096 Feb 10  2026 srv
dr-xr-xr-x  13 root root       0 Aug 22 11:51 sys
drwxrwxrwt  13 root root    4096 Aug 22 16:49 tmp
drwxr-xr-x  12 root root    4096 Feb 10  2026 usr
drwxr-xr-x  13 root root    4096 Aug 21 21:32 var

**`ls -la /`** — matches the FHS tree, plus a few WSL-specific extras worth flagging:

- **`bin.usr-is-merged`, `sbin.usr-is-merged`, `lib.usr-is-merged`** — leftover marker directories from the usr-merge migration we discussed. They're empty artifacts the conversion tool left behind as a "this has been merged" flag. Not used at runtime — ignore them, but if you ever see `*.usr-is-merged` on a real support ticket, now you know why it exists instead of panicking that something's duplicated.
- **`init`** — a WSL2-specific binary (not the classic `/sbin/init`). WSL2 runs its own lightweight init process to bootstrap the Linux VM before systemd takes over inside it.
- **`lost+found`** — only appears on `ext4`-formatted filesystems (your `/` is `/dev/sdd`, a real block device here). It's where the filesystem checker (`fsck`) dumps recovered file fragments after an unclean shutdown/crash. Empty and harmless 99% of the time — but if it's suddenly full of files, that's a signal there was a disk corruption event.
- **`opt`, `srv`, `snap`** — `/opt` is for self-contained third-party software (rarely used on Ubuntu); `/srv` is data served by this host (web/FTP roots — rarely used on a workstation); `/snap` is Ubuntu's Snap package mount points.

In [ ]:
dennie@HP:~/linux-tut$ df -h
Filesystem      Size  Used Avail Use% Mounted on
none            3.9G     0  3.9G   0% /usr/lib/modules/6.18.33.2-microsoft-standard-WSL2
none            3.9G  4.0K  3.9G   1% /mnt/wsl
drivers         195G   91G  104G  47% /usr/lib/wsl/drivers
/dev/sdd       1007G  3.2G  953G   1% /
none            3.9G   64K  3.9G   1% /mnt/wslg
none            3.9G     0  3.9G   0% /usr/lib/wsl/lib
rootfs          3.9G  2.8M  3.9G   1% /init
none            3.9G  468K  3.9G   1% /run
none            3.9G     0  3.9G   0% /run/lock
none            3.9G     0  3.9G   0% /run/shm
none            3.9G   80K  3.9G   1% /mnt/wslg/versions.txt
none            3.9G   80K  3.9G   1% /mnt/wslg/doc
C:\             195G   91G  104G  47% /mnt/c
D:\             737G  310G  427G  43% /mnt/d
tmpfs           787M   20K  787M   1% /run/user/1000

**`df -h`** — this is the interesting one for WSL:

- `/dev/sdd` on `/` — your actual Linux root, backed by that `.vhdx` virtual disk we mentioned. ~1TB allocated, 3.2G used.
- `/mnt/c`, `/mnt/d` — your real Windows drives, mounted via the `drvfs` driver (matches the sizes you'd see in Windows: 195G/104G free on C:, 737G/427G free on D:).
- `/mnt/wslg` — **WSLg**, the subsystem that lets WSL run Linux GUI apps and pipes them onto your Windows desktop. Not something you'll touch in support work, but good to recognize.
- `/run/user/1000` — a **per-user** runtime directory (tmpfs), `1000` being your UID. This is `$XDG_RUNTIME_DIR` — apps use it for user-scoped sockets/temp state, separate from the system-wide `/run` we covered.
- `drivers` mounted at `/usr/lib/wsl/drivers` — WSL2 passing through GPU drivers from Windows so Linux can do GPU-accelerated workloads.

> **WHAT TO LOOK FOR:** none of this is abnormal — this is what a healthy WSL2 Ubuntu `df -h` should look like. If a real ticket ever showed `/` at 100% Use, or `/dev/sdd` missing entirely, _that's_ when this table becomes a diagnosis tool instead of background info.

**READ THE OUTPUT:**

- `pwd` — your **current absolute path**. Everything in Linux is either absolute (starts with `/`) or relative (starts from where you are).
- `ls -la /` — this is the **root directory**. Every single top-level folder you see (`bin`, `etc`, `var`, `home`, `mnt`, `proc`, `sys`, `dev`...) has a _specific, standardized purpose_ — this is the Filesystem Hierarchy Standard (FHS), and we're about to walk through why each one exists.
- `df -h` — shows what's actually _mounted_ where. In WSL you'll see something like a `/` filesystem (the Linux side) and separate entries for `/mnt/c`, `/mnt/d` (your Windows drives, mounted in).

**WHAT TO LOOK FOR:** In the `df -h` output, note the `Filesystem` column for `/mnt/c` — it'll likely show something like `drvfs`. That's WSL's special driver for bridging into NTFS. This is why permissions on `/mnt/c` behave weirdly (everything shows `777`) — NTFS doesn't have Linux permission bits, so WSL fakes it. We'll hit this again in Pillar 3.

**GOTCHA (WSL-specific):** Don't confuze `df -h` output with Windows Explorer's "This PC" drive view. In WSL, your Linux root `/` is actually a virtual disk file living somewhere inside Windows (a `.vhdx` file) — but from inside Linux, it just looks like a normal root filesystem. You never need to think about the `.vhdx` unless you're doing WSL disk management from the Windows side.
